# 注意：本章代码兼容 trl >= 0.16。旧版 trl (< 0.12) 使用完全不同的 PPOTrainer API。

# 第7章：RLHF (PPO)

## 本章目标
- 理解 PPO (Proximal Policy Optimization) 在 RLHF 中的角色
- 理解 actor-critic 框架在 LLM 训练中的应用
- 使用 trl 的 PPOTrainer 完成 RLHF 训练

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch transformers trl peft datasets accelerate bitsandbytes
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## PPO in RLHF 速览

RLHF PPO 实际需要 **4 个模型**：

1. **Policy Model (Actor)**: 我们要优化的 LLM（SFT 后的模型）
2. **Value Model (Critic)**: 预测状态价值函数，用于计算 GAE (Generalized Advantage Estimation) advantage。通常与 Policy Model 共享 backbone
3. **Reward Model**: 给 (prompt, response) 打分
4. **Reference Model**: SFT 模型的冻结副本，用于计算 KL penalty

PPO 的核心思想：
- Policy Gradient: 鼓励高分回答，抑制低分回答
- Clipping: 限制策略更新幅度，防止训崩
- KL Penalty: 确保模型不会偏离 reference 太远（保持语言能力）
- GAE Advantage: Value Model 预测状态价值，结合实际 reward 计算 advantage，降低方差

参考：[InstructGPT](https://arxiv.org/abs/2203.02155), [PPO](https://arxiv.org/abs/1707.06347)

In [ ]:
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer
from trl import PPOTrainer, PPOConfig
from datasets import load_dataset
import torch

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Policy model (Actor)
policy_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

# Reference model (frozen)
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)
ref_model.eval()

# Reward model
reward_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=1, torch_dtype=torch.float16, device_map="auto"
)

# Value model (Critic) — 通常与 Policy Model 共享 backbone
# 这里用独立的 Value Head 示例；实际中可通过 value_model=policy_model 共享
value_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

print("四个模型加载完成: Policy, Reference, Reward, Value")

In [ ]:
# 新版 TRL PPOConfig
ppo_config = PPOConfig(
    output_dir="./ppo_output",
    per_device_train_batch_size=4,
    learning_rate=1e-5,
    num_ppo_epochs=2,        # 旧名 ppo_epochs
    kl_coef=0.1,
)

dataset = load_dataset("Anthropic/hh-rlhf", split="train[:1000]")

def extract_prompt(example):
    text = example["chosen"]
    if "Human:" in text:
        prompt = text.split("Human:")[1].split("Assistant:")[0].strip()
    else:
        prompt = text[:100]
    return {"query": prompt}

dataset = dataset.map(extract_prompt)

# 新版 TRL PPOTrainer — 使用标准 Trainer 风格接口
ppo_trainer = PPOTrainer(
    args=ppo_config,
    processing_class=tokenizer,       # 旧名 tokenizer
    model=policy_model,
    ref_model=ref_model,
    reward_model=reward_model,        # 新增必需参数
    train_dataset=dataset,            # 旧名 dataset
    value_model=value_model,          # 新增必需参数（通常与 policy 共享 backbone）
)
print("PPOTrainer 初始化完成")

In [ ]:
# 新版 TRL PPOTrainer 使用标准 Trainer 接口
# 调用 train() 即可完成完整的 PPO 训练流程：
#   1. 生成 response
#   2. 计算 reward（通过 reward_model）
#   3. 计算 GAE advantage（通过 value_model）
#   4. 执行 PPO clipped update

print("开始 PPO 训练（这可能需要几分钟）...")
ppo_trainer.train()
print("PPO 训练完成")

## 分析 PPO 训练指标

PPO 训练中需要关注的关键指标：
- **Reward**: 应该逐步上升（模型在学更好的回答）
- **KL Divergence**: 不应增长太快（模型偏离 reference 的程度）
- **Loss/Policy Loss**: 应该逐步下降

如果 KL divergence 增长太快，增大 `kl_coef`。
如果 reward 不涨，检查 Reward Model 的质量。

## 练习

1. 调整 `kl_coef` (0.01, 0.1, 1.0)，观察 KL divergence 和 reward 的变化
2. 增大 `ppo_epochs`，观察训练稳定性
3. 对比 RLHF 前后的生成结果

## 延伸阅读

- [InstructGPT](https://arxiv.org/abs/2203.02155)
- [PPO 论文](https://arxiv.org/abs/1707.06347)
- [trl PPOTrainer 文档](https://huggingface.co/docs/trl/ppo_trainer)
- [Deep RLHF 实践指南](https://huggingface.co/blog/rlhf)